In [1]:
# Install missing dependencies only (Kaggle usually already has torch/pandas).
import importlib
import subprocess
import sys

for pkg in ["polars"]:
    try:
        importlib.import_module(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])

In [2]:
import os
import time
import glob
import random
import math
import gc
from pathlib import Path
import numpy as np
import polars as pl
import pandas as pd
from collections import defaultdict
from contextlib import nullcontext
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast, GradScaler

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {DEVICE}")
print(f"PyTorch version: {torch.__version__}")

Device: cuda
PyTorch version: 2.10.0+cu128


In [3]:
!nvidia-smi

Tue May 26 00:58:45 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.105.08             Driver Version: 580.105.08     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX PRO 6000 Blac...    Off |   00000000:05:00.0 Off |                    0 |
| N/A   33C    P0             48W /  600W |       3MiB /  97887MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [ ]:
# Configuration
SEED = 42

class Config:
    embed_dim = 200
    # embed_dim = 128
    batch_size = 4112
    # batch_size = 8224
    lr = 0.001
    # maxlen = 50
    maxlen = 70
    num_epochs = 50
    num_heads = 4
    num_layers = 2
    dropout = 0.5
    warmup_steps = 500

config = Config()

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(SEED)

if torch.cuda.is_available():
    torch.backends.cudnn.benchmark = True
try:
    torch.set_float32_matmul_precision('high')
except Exception:
    pass

OUTPUT_DIR = "/kaggle/working/mbsrec_output/"
os.makedirs(OUTPUT_DIR, exist_ok=True)

LOG_FILE = os.path.join(OUTPUT_DIR, "training_log.txt")

In [5]:
# Data paths (simple, fixed for this Kaggle dataset)
from pathlib import Path

DATA_PATH = Path("/kaggle/input/datasets/trungnguyen2710t/otto-ptit-filter")
TRAIN_PARQUET = DATA_PATH / "train.parquet"
VAL_PARQUET = DATA_PATH / "val.parquet"
TEST_PARQUET  = DATA_PATH / "test.parquet"

if not TRAIN_PARQUET.exists() or not VAL_PARQUET.exists():
    raise FileNotFoundError(f"Missing files in {DATA_PATH}")

print(f"Train : {TRAIN_PARQUET}")
print(f"Val   : {VAL_PARQUET}")

# Keep same variable names used in later cells
train_files = [str(TRAIN_PARQUET)]
test_files = [str(VAL_PARQUET)]  # dùng val làm tập eval/test nội bộ

print(f"Using DATA_PATH: {DATA_PATH}")
print(f"Train: {train_files[0]}")
print(f"Val/Test-internal: {test_files[0]}")

Train : /kaggle/input/datasets/trungnguyen2710t/otto-ptit-filter/train.parquet
Val   : /kaggle/input/datasets/trungnguyen2710t/otto-ptit-filter/val.parquet
Using DATA_PATH: /kaggle/input/datasets/trungnguyen2710t/otto-ptit-filter
Train: /kaggle/input/datasets/trungnguyen2710t/otto-ptit-filter/train.parquet
Val/Test-internal: /kaggle/input/datasets/trungnguyen2710t/otto-ptit-filter/val.parquet


In [6]:
import os, gc, shutil
import polars as pl
import numpy as np
 
type_weight_map = {'clicks': 0.1, 'carts': 0.3, 'orders': 0.6}
 
def normalize_event_type(raw_type):
    if isinstance(raw_type, (int, np.integer)):
        return {0: 'clicks', 1: 'carts', 2: 'orders'}.get(int(raw_type), 'clicks')
    raw = str(raw_type).strip().lower()
    if raw in {'0', '1', '2'}:
        return {'0': 'clicks', '1': 'carts', '2': 'orders'}[raw]
    return raw
 
def _collect(lf):
    try:
        return lf.collect(engine="gpu")
    except Exception:
        return lf.collect()

In [8]:
# TRAIN
# Luồng: scan parquet → chunk → filter → lưu (session, items, types) ra disk → merge → load lại
def process_train_data_chunked(parquet_path, chunk_size=500_000):
    TMP_DIR    = "/kaggle/working/_tmp_train_mbsrec"
    FINAL_PATH = "/kaggle/working/train_mbsrec.parquet"
    os.makedirs(TMP_DIR, exist_ok=True)
 
    lf = pl.scan_parquet(str(parquet_path)).select(["session", "aid", "ts", "type"])
 
    # itemnum: aid gốc từ 0, shift +1 → itemnum = aid_max + 1
    itemnum = int(_collect(lf.select(pl.col("aid").max()))["aid"][0]) + 1
    print(f"[Train] itemnum = {itemnum}", flush=True)
 
    all_sessions = _collect(lf.select("session").unique())["session"].to_list()
    total    = len(all_sessions)
    n_chunks = (total + chunk_size - 1) // chunk_size
    print(f"[Train] {total:,} sessions → {n_chunks} chunks", flush=True)
 
    dropped_no_buy    = 0
    dropped_too_short = 0
    kept_total        = 0
 
    for i in range(0, total, chunk_size):
        chunk_idx      = i // chunk_size + 1
        chunk_sessions = all_sessions[i : i + chunk_size]
 
        chunk_df = _collect(
            lf.filter(pl.col("session").is_in(chunk_sessions))
            .sort(["session", "ts"])
            .group_by("session")
            .agg([
                (pl.col("aid") + 1).sort_by(pl.col("ts")).alias("items"),  # shift +1
                pl.col("type").sort_by(pl.col("ts")).alias("types"),
                pl.col("type").filter(pl.col("type") == "orders").count().alias("order_count"),
                pl.col("aid").count().alias("session_len"),
            ])
            .filter(
                (pl.col("order_count") > 0) &
                (pl.col("session_len") > 1)
            )           
            .select(["session", "items", "types"])
        )
 
        dropped_no_buy    += len(chunk_sessions) - chunk_df.height
        dropped_too_short  = 0  
 
        rows_out = []
        for row in chunk_df.iter_rows(named=True):
            session     = int(row["session"])
            items       = [int(x) for x in row["items"]]  # đã shift +1
            event_types = [{'clicks': 0, 'carts': 1, 'orders': 2}.get(normalize_event_type(t), 0) for t in row["types"]]
            rows_out.append({
                "session": session,
                "items": items,
                "types": event_types
            })
 
        if rows_out:
            pl.DataFrame({
                "session": [r["session"] for r in rows_out],
                "items": [r["items"] for r in rows_out],
                "types": [r["types"] for r in rows_out]
            }).write_parquet(f"{TMP_DIR}/chunk_{chunk_idx:03d}.parquet")

        kept_total += len(rows_out)
        del chunk_df, rows_out
        gc.collect()
        print(f"  Chunk {chunk_idx}/{n_chunks}: {kept_total:,} kept → disk", flush=True)
 
    del lf
    gc.collect()
 
    print("[Train] Merging via sink_parquet (no RAM)...", flush=True)
    pl.scan_parquet(f"{TMP_DIR}/*.parquet").sink_parquet(FINAL_PATH)
    shutil.rmtree(TMP_DIR)
    gc.collect()
    print(f"[Train] Saved → {FINAL_PATH}", flush=True)
 
    lf_final = pl.scan_parquet(FINAL_PATH)
    train_item_ids = set(
        lf_final.select(pl.col("items").explode()).unique().collect()["items"].to_list()
    )
    gc.collect()
    print(f"[Train] train_item_ids: {len(train_item_ids):,} unique items", flush=True)
 
    train_df = lf_final.select(["session", "items", "types"]).collect()
    usernum  = train_df.height
    gc.collect()
 
    print(f"[Train] Dropped no orders: {dropped_no_buy:,}", flush=True)
    print(f"[Train] Kept: {usernum:,} sessions  |  Items: {itemnum:,}", flush=True)
    return train_df, usernum, itemnum, train_item_ids

In [10]:
def process_val_data_chunked(parquet_path, train_item_ids, chunk_size=500_000):
    TMP_DIR    = "/kaggle/working/_tmp_val_mbsrec"
    FINAL_PATH = "/kaggle/working/val_mbsrec.parquet"
    os.makedirs(TMP_DIR, exist_ok=True)
 
    lf = pl.scan_parquet(str(parquet_path)).select(["session", "aid", "ts", "type"])
 
    df_val = _collect(
        lf.sort(["session", "ts"])
        .group_by("session")
        .agg([
            (pl.col("aid") + 1).sort_by(pl.col("ts")).alias("items"),
            pl.col("type").sort_by(pl.col("ts")).alias("types"),
        ])
        .filter(pl.col("types").list.last() == "orders")
        .filter(pl.col("items").list.len() > 1)
    )
    del lf
    gc.collect()
    print(f"[Val] After basic filter: {df_val.height:,} sessions", flush=True)
 
    train_ids_series = pl.Series("aid", list(train_item_ids), dtype=pl.Int64)
    df_val = df_val.filter(
        pl.col("items").list.eval(
            pl.element().cast(pl.Int64).is_in(train_ids_series)
        ).list.all()
    )
    gc.collect()
    print(f"[Val] After OOV filter: {df_val.height:,} sessions", flush=True)
 
    all_sessions = df_val["session"].to_list()
    total    = len(all_sessions)
    n_chunks = (total + chunk_size - 1) // chunk_size
 
    dropped_not_last  = 0
    dropped_too_short = 0
    kept_total        = 0
 
    for i in range(0, total, chunk_size):
        chunk_idx      = i // chunk_size + 1
        chunk_sessions = set(all_sessions[i : i + chunk_size])
 
        chunk_df = df_val.filter(pl.col("session").is_in(list(chunk_sessions)))
 
        rows_out = []
        for row in chunk_df.iter_rows(named=True):
            session     = int(row["session"])
            items       = [int(x) for x in row["items"]]
            event_types = [{'clicks': 0, 'carts': 1, 'orders': 2}.get(normalize_event_type(t), 0) for t in row["types"]]
 
            last_order_idx = None
            for idx in range(len(event_types) - 1, -1, -1):
                if event_types[idx] == 2:
                    last_order_idx = idx
                    break
            if last_order_idx is None or last_order_idx != len(items) - 1:
                dropped_not_last += 1
                continue
 
            context = items[:last_order_idx]
            context_types = event_types[:last_order_idx]
            target  = items[last_order_idx]
 
            if len(context) < 1:
                dropped_too_short += 1
                continue
 
            rows_out.append({
                "session": session,
                "context": context,
                "context_types": context_types,
                "target" : target,
            })
 
        if rows_out:
            pl.DataFrame({
                "session": [r["session"] for r in rows_out],
                "context": [r["context"] for r in rows_out],
                "context_types": [r["context_types"] for r in rows_out],
                "target" : [r["target"]  for r in rows_out],
            }).write_parquet(f"{TMP_DIR}/chunk_{chunk_idx:03d}.parquet")
 
        kept_total += len(rows_out)
        del chunk_df, rows_out
        gc.collect()
        print(f"  Chunk {chunk_idx}/{n_chunks}: {kept_total:,} kept → disk", flush=True)
 
    del df_val
    gc.collect()
 
    print("[Val] Merging via sink_parquet (no RAM)...", flush=True)
    pl.scan_parquet(f"{TMP_DIR}/*.parquet").sink_parquet(FINAL_PATH)
    shutil.rmtree(TMP_DIR)
    gc.collect()
    print(f"[Val] Saved → {FINAL_PATH}", flush=True)
 
    val_df = pl.scan_parquet(FINAL_PATH).collect()
    gc.collect()
 
    print(f"[Val] Dropped not-last-order: {dropped_not_last:,}  |  too short: {dropped_too_short:,}", flush=True)
    print(f"[Val] Kept: {val_df.height:,} sessions", flush=True)
    return val_df

In [11]:
# ─────────────────────────────────────────────────────────────────────────────
# Build user_train / val_seq / val_gt từ df đã load
# ─────────────────────────────────────────────────────────────────────────────
def build_user_train(train_df):
    user_train_items = {}
    user_train_types = {}
    print(f"[Train] Building user_train from {train_df.height:,} sessions...", flush=True)
    for i, row in enumerate(train_df.iter_rows(named=True)):
        session = int(row["session"])
        user_train_items[session] = [int(x) for x in row["items"]]
        user_train_types[session] = [int(x) for x in row["types"]]
        if i % 200_000 == 0:
            print(f"  {i:,}/{train_df.height:,}...", flush=True)
    print(f"[Train] user_train: {len(user_train_items):,} users", flush=True)
    return user_train_items, user_train_types
 
 
def build_val_structures(val_df):
    val_seq = {}
    val_seq_types = {}
    val_gt  = {}
    print(f"[Val] Building val_seq / val_gt from {val_df.height:,} sessions...", flush=True)
    for i, row in enumerate(val_df.iter_rows(named=True)):
        session = int(row["session"])
        val_seq[session] = [int(x) for x in row["context"]]
        val_seq_types[session] = [int(x) for x in row["context_types"]]
        val_gt[session]  = [int(row["target"])]
        if i % 100_000 == 0:
            print(f"  {i:,}/{val_df.height:,}...", flush=True)
    print(f"[Val] val_seq: {len(val_seq):,}  |  val_gt: {len(val_gt):,}", flush=True)
    return val_seq, val_seq_types, val_gt

In [12]:
# ─────────────────────────────────────────────────────────────────────────────
# MAIN — tách thành cell riêng để tránh Kaggle timeout
# ─────────────────────────────────────────────────────────────────────────────
 
# ── Cell 1 ────────────────────────────────────────────────────────────────
CHUNK_SIZE = 500_000  # giảm xuống 300_000 nếu vẫn OOM
 
print("Partitioning training data...")
train_df, usernum, itemnum, train_item_ids = process_train_data_chunked(
    TRAIN_PARQUET, chunk_size=CHUNK_SIZE
)
gc.collect()
 
# ── Cell 2 ────────────────────────────────────────────────────────────────
user_train_items, user_train_types = build_user_train(train_df)
del train_df
gc.collect()
print("✓ user_train built", flush=True)
 
# ── Cell 3 ────────────────────────────────────────────────────────────────
print("\nPartitioning validation data...")
val_df = process_val_data_chunked(
    VAL_PARQUET, train_item_ids, chunk_size=CHUNK_SIZE
)
del train_item_ids
gc.collect()
 
# ── Cell 4 ────────────────────────────────────────────────────────────────
val_seq, val_seq_types, val_gt = build_val_structures(val_df)
del val_df
gc.collect()
print("✓ val_seq / val_gt built", flush=True)
 
print(f"\nTrain users: {usernum}  |  Items: {itemnum}")
print(f"Val sessions (after filtering): {len(val_seq)}")

Partitioning training data...
[Train] itemnum = 1855603
[Train] 6,138,317 sessions → 13 chunks
  Chunk 1/13: 107,152 kept → disk
  Chunk 2/13: 214,077 kept → disk
  Chunk 3/13: 320,546 kept → disk
  Chunk 4/13: 427,617 kept → disk
  Chunk 5/13: 534,699 kept → disk
  Chunk 6/13: 641,783 kept → disk
  Chunk 7/13: 748,558 kept → disk
  Chunk 8/13: 855,644 kept → disk
  Chunk 9/13: 962,901 kept → disk
  Chunk 10/13: 1,069,993 kept → disk
  Chunk 11/13: 1,177,306 kept → disk
  Chunk 12/13: 1,284,068 kept → disk
  Chunk 13/13: 1,313,592 kept → disk
[Train] Merging via sink_parquet (no RAM)...
[Train] Saved → /kaggle/working/train_mbsrec.parquet
[Train] train_item_ids: 1,604,444 unique items
[Train] Dropped no orders: 4,824,725
[Train] Kept: 1,313,592 sessions  |  Items: 1,855,603
[Train] Building user_train from 1,313,592 sessions...
  0/1,313,592...
  200,000/1,313,592...
  400,000/1,313,592...
  600,000/1,313,592...
  800,000/1,313,592...
  1,000,000/1,313,592...
  1,200,000/1,313,592...
[

/tmp/ipykernel_163/2582974077.py:23: DeprecationWarning: `is_in` with a collection of the same datatype is ambiguous and deprecated.
Please use `implode` to return to previous behavior.

See https://github.com/pola-rs/polars/issues/22149 for more information.
  df_val = df_val.filter(



  Chunk 1/1: 86,727 kept → disk
[Val] Merging via sink_parquet (no RAM)...
[Val] Saved → /kaggle/working/val_mbsrec.parquet
[Val] Dropped not-last-order: 0  |  too short: 0
[Val] Kept: 86,727 sessions
[Val] Building val_seq / val_gt from 86,727 sessions...
  0/86,727...
[Val] val_seq: 86,727  |  val_gt: 86,727
✓ val_seq / val_gt built

Train users: 1313592  |  Items: 1855603
Val sessions (after filtering): 86727


In [13]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math


# ─────────────────────────────────────────────────────────────────────────────
# 1. Positional Encoding  (fixed sinusoidal, identical to original modules.py)
#    Original: positional_encoding(dim, sentence_length) – learnable embedding
#    table is NOT used; a fixed tensor is added to the sequence.
# ─────────────────────────────────────────────────────────────────────────────
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=200):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.pow(
            10000.0,
            torch.arange(0, d_model, 2, dtype=torch.float) * (2.0 / d_model)
        )
        pe[:, 0::2] = torch.sin(position / div_term)
        pe[:, 1::2] = torch.cos(position / div_term)
        # shape: (1, max_len, d_model) – broadcast over batch
        self.register_buffer("pe", pe.unsqueeze(0))

    def forward(self, x):
        # x: (B, T, D)  – add positional signal, no dropout (original adds directly)
        return x + self.pe[:, :x.size(1), :]


# ─────────────────────────────────────────────────────────────────────────────
# 2. Layer Normalisation  (direct port of normalize() in modules.py)
# ─────────────────────────────────────────────────────────────────────────────
class LayerNorm(nn.Module):
    def __init__(self, d_model, eps=1e-8):
        super().__init__()
        self.gamma = nn.Parameter(torch.ones(d_model))
        self.beta  = nn.Parameter(torch.zeros(d_model))
        self.eps   = eps

    def forward(self, x):
        mean = x.mean(dim=-1, keepdim=True)
        var  = x.var(dim=-1, keepdim=True, unbiased=False)
        return self.gamma * (x - mean) / (var + self.eps).sqrt() + self.beta


# ─────────────────────────────────────────────────────────────────────────────
# 3. Multi-Head Attention  (mirrors multihead_attention() in modules.py)
#    Key differences from previous version:
#      - key_mask  = sign(sum(abs(keys)))  (vector-level, not token-id check)
#      - query_mask applied AFTER softmax  (multiply attention weights)
#      - NO residual or norm inside – handled by TransformerBlock
# ─────────────────────────────────────────────────────────────────────────────
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads, dropout=0.0):
        super().__init__()
        assert d_model % num_heads == 0
        self.d_model   = d_model
        self.num_heads = num_heads
        self.d_head    = d_model // num_heads

        self.W_Q = nn.Linear(d_model, d_model, bias=False)
        self.W_K = nn.Linear(d_model, d_model, bias=False)
        self.W_V = nn.Linear(d_model, d_model, bias=False)
        self.dropout = nn.Dropout(dropout)

    def forward(self, queries, keys, causality=True):
        B, T_q, _ = queries.shape
        B, T_k, _ = keys.shape
        h = self.num_heads

        Q = self.W_Q(queries)   # (B, T_q, D)
        K = self.W_K(keys)      # (B, T_k, D)
        V = self.W_V(keys)      # (B, T_k, D)

        # Split heads: (B, h, T, d_head)
        def split_heads(t):
            return t.view(B, -1, h, self.d_head).transpose(1, 2)

        Q, K, V = split_heads(Q), split_heads(K), split_heads(V)

        # Scaled dot-product
        scale   = self.d_head ** 0.5
        scores  = torch.matmul(Q, K.transpose(-2, -1)) / scale  # (B, h, T_q, T_k)

        NEG_INF = torch.finfo(scores.dtype).min  # dtype-safe: works with both fp32 and fp16

        # Key masking  – original: sign(reduce_sum(abs(keys), axis=-1))
        key_mask = keys.abs().sum(dim=-1).sign()           # (B, T_k)  values 0 or 1
        key_mask = key_mask.unsqueeze(1).unsqueeze(2)      # (B, 1, 1, T_k)
        scores   = scores.masked_fill(key_mask == 0, NEG_INF)

        # Causal mask  (lower-triangular: attend only to past & current)
        if causality:
            causal = torch.tril(torch.ones(T_q, T_k, device=queries.device))
            scores = scores.masked_fill(causal.unsqueeze(0).unsqueeze(0) == 0, NEG_INF)

        # Softmax
        attn = F.softmax(scores, dim=-1)
        attn = torch.nan_to_num(attn, nan=0.0)

        # Query masking  – original: sign(reduce_sum(abs(queries), axis=-1))
        query_mask = queries.abs().sum(dim=-1).sign()      # (B, T_q)
        query_mask = query_mask.unsqueeze(1).unsqueeze(-1) # (B, 1, T_q, 1)
        attn = attn * query_mask

        attn    = self.dropout(attn)
        context = torch.matmul(attn, V)                    # (B, h, T_q, d_head)
        context = context.transpose(1, 2).contiguous().view(B, T_q, self.d_model)
        return context


# ─────────────────────────────────────────────────────────────────────────────
# 4. Feed-Forward Network  (mirrors feedforward() in modules.py)
#    Original uses Conv1d(kernel_size=1) which is equivalent to a per-position
#    Linear.  Structure: Linear(ReLU) → Dropout → Linear → Dropout → +residual
#    NO LayerNorm inside – normalisation is applied before calling this in the block.
# ─────────────────────────────────────────────────────────────────────────────
class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff, dropout=0.0):
        super().__init__()
        self.linear1 = nn.Linear(d_model, d_ff)
        self.linear2 = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = self.dropout(F.relu(self.linear1(x)))
        out = self.dropout(self.linear2(out))
        return out + x   # residual connection (identical to original outputs += inputs)


# ─────────────────────────────────────────────────────────────────────────────
# 5. Transformer Block  (mirrors the block loop in model.py)
#    Order in original:
#      a) self-attention  (residual inside multihead_attention when res=True)
#      b) feedforward(normalize(seq))   <- Pre-Norm on the attention output
#      c) seq *= mask
#    There is NO post-attention LayerNorm; the norm goes into feedforward's input.
# ─────────────────────────────────────────────────────────────────────────────
class TransformerBlock(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout=0.0):
        super().__init__()
        self.attention = MultiHeadAttention(d_model, num_heads, dropout)
        self.norm      = LayerNorm(d_model)          # Pre-Norm before FF
        self.ff        = FeedForward(d_model, d_ff, dropout)

    def forward(self, x, padding_mask):
        # a) Self-attention + residual  (res=True in original)
        attn_out = self.attention(x, x, causality=True)
        x = x + attn_out                             # residual (same as original res=True)

        # b) Pre-Norm then Feed-Forward (feedforward(normalize(seq)) in original)
        x = self.ff(self.norm(x))

        # c) Re-apply padding mask  (seq *= mask in original)
        x = x * padding_mask                         # padding_mask: (B, T, 1) float

        return x


# ─────────────────────────────────────────────────────────────────────────────
# 6. MBSRecModel – Encoder
#    Original fusion strategy (model.py lines 32-39 & 121-133):
#      seq_emb   = Concat([item_emb * sqrt(d), cxt_linear(beh_vec)]) -> feat_emb_linear
#      target_emb = Concat([item_emb,            cxt_linear(beh_vec)]) -> feat_emb_linear  (reuse=True)
#    The SAME cxt_linear and feat_emb_linear are shared between seq and target paths.
# ─────────────────────────────────────────────────────────────────────────────
class MBSRecModel(nn.Module):
    def __init__(self, num_items, num_behaviors, d_model, maxlen,
                 num_heads, num_layers, dropout=0.1):
        super().__init__()
        self.d_model = d_model

        # Item embedding table  (zero_pad=True: row 0 is forced to zeros)
        self.item_emb = nn.Embedding(num_items + 1, d_model, padding_idx=0)

        # Shared behaviour linear  (cxt_emb in original, reuse=True)
        self.cxt_linear  = nn.Linear(num_behaviors, d_model, bias=False)

        # Shared feature fusion  (feat_emb in original, reuse=True)
        # Input is concat of item_emb + cxt_emb → d_model
        self.feat_linear = nn.Linear(d_model * 2, d_model, bias=False)

        # Positional encoding (fixed sinusoidal, added after dropout)
        self.pos_enc = PositionalEncoding(d_model, maxlen)

        self.dropout = nn.Dropout(dropout)

        self.blocks = nn.ModuleList([
            TransformerBlock(d_model, num_heads, d_model, dropout)
            for _ in range(num_layers)
        ])

        self.final_norm = LayerNorm(d_model)  # normalize(seq) at end of model.py line 113

        self._init_weights()

    def _init_weights(self):
        nn.init.normal_(self.item_emb.weight, std=0.01)
        self.item_emb.weight.data[0] = 0   # enforce zero-pad row
        nn.init.normal_(self.cxt_linear.weight,  std=0.01)
        nn.init.normal_(self.feat_linear.weight, std=0.01)

    def encode_items(self, item_ids, beh_vecs):
        # Fuse item embedding and behaviour context the same way for BOTH
        # the input sequence and the target items (shared weights).
        # item_ids : (...,)    int tensor
        # beh_vecs : (..., num_behaviors)  float tensor
        item_e = self.item_emb(item_ids)           # (..., D)
        cxt_e  = self.cxt_linear(beh_vecs)         # (..., D)
        fused  = self.feat_linear(
            torch.cat([item_e, cxt_e], dim=-1)     # (..., 2D) -> (..., D)
        )
        return fused

    def forward(self, seq_ids, seq_beh):
        # seq_ids : (B, T)  int
        # seq_beh : (B, T, num_behaviors)  float
        # Padding mask: (B, T, 1) float  –  1 where item != 0
        pad_mask = (seq_ids != 0).float().unsqueeze(-1)   # (B, T, 1)

        # Item + context fusion (original lines 22-39)
        x = self.encode_items(seq_ids, seq_beh)            # (B, T, D)

        # Scale embedding like original: outputs * sqrt(num_units)  [line 128]
        x = x * (self.d_model ** 0.5)

        # Positional encoding + dropout + mask (lines 53, 71-74)
        x = self.pos_enc(x)
        x = self.dropout(x)
        x = x * pad_mask

        # Transformer blocks  (lines 78-111)
        for block in self.blocks:
            x = block(x, pad_mask)

        # Final layer norm  (line 113: self.seq = normalize(self.seq))
        x = self.final_norm(x)

        return x   # (B, T, D)


# ─────────────────────────────────────────────────────────────────────────────
# 7. MBSRec  – wrapper used during training and inference
# ─────────────────────────────────────────────────────────────────────────────
class MBSRec(nn.Module):
    def __init__(self, num_items, num_behaviors=3, embed_dim=64,
                 maxlen=50, num_heads=2, num_layers=2, dropout=0.1):
        super().__init__()
        self.encoder = MBSRecModel(
            num_items, num_behaviors, embed_dim, maxlen,
            num_heads, num_layers, dropout
        )

    def target_item_embedding(self, item_ids, beh_vecs):
        # Embed target / candidate items through the SAME shared layers
        # (cxt_linear + feat_linear) as the sequence encoder.
        # PyTorch equivalent of reuse=True in the original TF code.
        return self.encoder.encode_items(item_ids, beh_vecs)

    def forward(self, seq_ids, seq_beh):
        return self.encoder(seq_ids, seq_beh)   # (B, T, D)


print("MBSRec Model defined  (aligned with original TF repo architecture)")


MBSRec Model defined  (aligned with original TF repo architecture)


In [14]:
# Dataset and Sampler

def random_neq(l, r, s):
    t = np.random.randint(l, r)
    while t in s:
        t = np.random.randint(l, r)
    return t

class MBSRecDataset(Dataset):
    def __init__(self, user_train_items, user_train_types, usernum, itemnum, maxlen, Beh_num=3):
        self.user_train_items = user_train_items
        self.user_train_types = user_train_types
        self.usernum = usernum
        self.itemnum = itemnum
        self.maxlen = maxlen
        self.Beh_num = Beh_num
        self.users = list(user_train_items.keys())
        
        self.type_to_onehot = [
            [1.0, 0.0, 0.0], # clicks
            [0.0, 1.0, 0.0], # carts
            [0.0, 0.0, 1.0], # orders
        ]
        self.type_to_weight = [0.1, 0.2, 0.7]
    
    def __len__(self):
        return len(self.users)
    
    def __getitem__(self, idx):
        user = self.users[idx]
        
        if len(self.user_train_items[user]) <= 1:
            return self.__getitem__((idx + 1) % len(self.users))
        
        seq = np.zeros(self.maxlen, dtype=np.int32)
        pos = np.zeros(self.maxlen, dtype=np.int32)
        neg = np.zeros(self.maxlen, dtype=np.int32)
        beh_seq = np.zeros((self.maxlen, self.Beh_num), dtype=np.float32)
        beh_pos = np.zeros((self.maxlen, self.Beh_num), dtype=np.float32)
        pos_weight = np.zeros(self.maxlen, dtype=np.float32)
        neg_weight = np.ones(self.maxlen, dtype=np.float32)

        items = self.user_train_items[user]
        types = self.user_train_types[user]

        nxt_item = items[-1]
        nxt_type = types[-1]
        idx_pos = self.maxlen - 1
        
        rated = set(items)
        
        for i in range(len(items) - 2, -1, -1):
            curr_item = items[i]
            curr_type = types[i]

            seq[idx_pos] = curr_item
            pos[idx_pos] = nxt_item
            neg[idx_pos] = random_neq(1, self.itemnum + 1, rated)
            
            beh_seq[idx_pos] = self.type_to_onehot[curr_type]
            beh_pos[idx_pos] = self.type_to_onehot[nxt_type]
            pos_weight[idx_pos] = self.type_to_weight[nxt_type]
            
            nxt_item = curr_item
            nxt_type = curr_type
            idx_pos -= 1
            if idx_pos == -1:
                break
        
        return (
            torch.LongTensor(seq),
            torch.LongTensor(pos),
            torch.LongTensor(neg),
            torch.FloatTensor(beh_seq),
            torch.FloatTensor(beh_pos),
            torch.FloatTensor(pos_weight),
            torch.FloatTensor(neg_weight)
        )

print("Dataset class defined")

Dataset class defined


In [15]:
# Loss Function

class WeightedBPRLoss(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, pos_logits, neg_logits, pos_weight, neg_weight, mask):
        mask = mask.float()
        pos_weight = pos_weight.float() * mask
        neg_weight = neg_weight.float() * mask

        pos_term = F.softplus(-pos_logits) * pos_weight
        neg_term = F.softplus(neg_logits) * neg_weight

        denom = mask.sum().clamp_min(1.0)
        loss = (pos_term + neg_term).sum() / denom
        loss = torch.nan_to_num(loss, nan=0.0, posinf=1e4, neginf=0.0)
        return loss

print("Loss function defined")

Loss function defined


In [16]:
# Evaluation Metrics

def compute_dcg(scores, k):
    scores = scores[:k]
    return sum((2**rel - 1) / math.log2(idx + 2) for idx, rel in enumerate(scores))

def compute_ndcg(rel_items, pred_items, k):
    scores = [1 if item in rel_items else 0 for item in pred_items[:k]]
    ideal_scores = [1] * min(len(rel_items), k) + [0] * max(0, k - len(rel_items))
    idcg = compute_dcg(ideal_scores, k)
    if idcg == 0:
        return 0.0
    return compute_dcg(scores, k) / idcg

def compute_recall(rel_items, pred_items, k):
    if len(rel_items) == 0:
        return 0.0
    pred_k  = set(pred_items[:k])
    rel_set = set(rel_items)
    return len(pred_k & rel_set) / len(rel_set)

def compute_hit_rate(rel_items, pred_items, k):
    pred_k  = set(pred_items[:k])
    rel_set = set(rel_items)
    return 1.0 if len(pred_k & rel_set) > 0 else 0.0

def build_eval_plan(val_seq, val_gt, itemnum,
                    max_users=2000, neg_samples=99, seed=2026):
    rng = np.random.default_rng(seed)

    eligible = [u for u in val_seq
                if len(val_seq[u]) >= 1 and len(val_gt.get(u, [])) >= 1]
    eligible = sorted(eligible)

    if len(eligible) > max_users:
        idx        = rng.choice(len(eligible), size=max_users, replace=False)
        eval_users = sorted([eligible[i] for i in idx])
    else:
        eval_users = eligible

    eval_negatives = {}
    for u in eval_users:
        rel_item = val_gt[u][0]
        rated    = set(val_seq[u])
        rated.add(0)

        negatives    = []
        max_attempts = max(neg_samples * 50, 1000)
        attempts     = 0
        while len(negatives) < neg_samples and attempts < max_attempts:
            t = int(rng.integers(1, itemnum + 1))
            attempts += 1
            if t == rel_item or t in rated or t in negatives:
                continue
            negatives.append(t)

        cursor = 1
        while len(negatives) < neg_samples and cursor <= itemnum:
            if cursor != rel_item and cursor not in rated and cursor not in negatives:
                negatives.append(cursor)
            cursor += 1

        eval_negatives[u] = negatives

    return eval_users, eval_negatives

def evaluate_full(model, val_seq, val_seq_types, val_gt, itemnum, config, device,
                  ks=[10, 20], eval_users=None, eval_negatives=None):
    model.eval()
    BEH_NUM    = 3
    TARGET_BEH = [0, 0, 1]   # 'orders'
    type_to_onehot = [
        [1.0, 0.0, 0.0], # clicks
        [0.0, 1.0, 0.0], # carts
        [0.0, 0.0, 1.0], # orders
    ]

    metrics = {
        'Recall@10': 0.0, 'Recall@20': 0.0,
        'NDCG@10':   0.0, 'NDCG@20':   0.0,
        'HR@10':     0.0, 'HR@20':      0.0,
    }
    valid_users = 0
    users = eval_users if eval_users is not None else list(val_seq.keys())

    with torch.no_grad():
        for u in users:
            if u not in val_seq or len(val_seq[u]) < 1:
                continue
            if u not in val_gt or len(val_gt[u]) < 1:
                continue

            seq     = np.zeros(config.maxlen, dtype=np.int32)
            beh_seq = np.zeros((config.maxlen, BEH_NUM), dtype=np.float32)
            idx_pos = config.maxlen - 1
            
            items = val_seq[u]
            types = val_seq_types[u]
            for i in range(len(items) - 1, -1, -1):
                seq[idx_pos]     = items[i]
                beh_seq[idx_pos] = type_to_onehot[types[i]]
                idx_pos -= 1
                if idx_pos == -1:
                    break

            rel_item  = val_gt[u][0]
            rel_items = [rel_item]
            rated     = set(val_seq[u])
            rated.add(0)

            candidates = [rel_item]
            if eval_negatives is not None and u in eval_negatives:
                candidates.extend(eval_negatives[u][:99])
            else:
                while len(candidates) < 100:
                    t = np.random.randint(1, itemnum + 1)
                    if t in rated or t in candidates:
                        continue
                    candidates.append(t)

            candidate_context = [TARGET_BEH for _ in candidates]

            seq_tensor      = torch.LongTensor(seq).unsqueeze(0).to(device)
            beh_tensor      = torch.FloatTensor(beh_seq).unsqueeze(0).to(device)
            enc_out         = model(seq_tensor, beh_tensor)
            seq_emb         = enc_out[:, -1, :]

            cand_tensor     = torch.LongTensor(candidates).to(device)
            cand_ctx_tensor = torch.FloatTensor(candidate_context).to(device)
            cand_emb        = model.target_item_embedding(cand_tensor, cand_ctx_tensor)

            scores         = torch.matmul(seq_emb, cand_emb.T).squeeze(0).cpu().numpy()
            scores         = np.nan_to_num(scores, nan=-1e9, posinf=1e9, neginf=-1e9)
            ranked_indices = scores.argsort()[::-1]
            pred_items     = [candidates[i] for i in ranked_indices]

            for k in ks:
                metrics[f'Recall@{k}'] += compute_recall(rel_items,  pred_items, k)
                metrics[f'NDCG@{k}']   += compute_ndcg(rel_items,    pred_items, k)
                metrics[f'HR@{k}']     += compute_hit_rate(rel_items, pred_items, k)

            valid_users += 1

    if valid_users > 0:
        for key in metrics:
            metrics[key] /= valid_users

    return metrics, valid_users

print("Evaluation metrics defined")

Evaluation metrics defined


In [ ]:
# Initialize Model
print("\n" + "="*70)
print("Initializing MBSRec Model")
print("="*70 + "\n")

model = MBSRec(
    num_items=itemnum,
    num_behaviors=3,
    embed_dim=config.embed_dim,
    maxlen=config.maxlen,
    num_heads=config.num_heads,
    num_layers=config.num_layers,
    dropout=config.dropout
).to(DEVICE)

dataset = MBSRecDataset(user_train_items, user_train_types, usernum, itemnum, config.maxlen)
dataloader = DataLoader(
    dataset,
    batch_size=config.batch_size,
    shuffle=True,
    num_workers=0,
    pin_memory=(DEVICE == 'cuda')
)

optimizer = torch.optim.AdamW(model.parameters(), lr=config.lr, weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=config.num_epochs)
criterion = WeightedBPRLoss()
try:
    scaler = GradScaler(device='cuda', enabled=(DEVICE == 'cuda'))
except TypeError:
    scaler = GradScaler(enabled=(DEVICE == 'cuda'))

EVAL_MAX_USERS = 2000
EVAL_NEGATIVES = 99
EVAL_SEED      = 2026

fixed_eval_users, fixed_eval_negatives = build_eval_plan(
    val_seq,
    val_gt,
    itemnum,
    max_users=EVAL_MAX_USERS,
    neg_samples=EVAL_NEGATIVES,
    seed=EVAL_SEED,
)

total_params = sum(p.numel() for p in model.parameters())
print(f"Model parameters: {total_params:,}")
print(f"Train users: {usernum}  |  Items: {itemnum}")
print(f"Val sessions (after filtering): {len(val_seq)}")
print(f"Embed dim: {config.embed_dim}, Heads: {config.num_heads}, Layers: {config.num_layers}")
print(f"Maxlen: {config.maxlen}, Batch: {config.batch_size}, Epochs: {config.num_epochs}")
print(f"Fixed eval users: {len(fixed_eval_users)}, neg/user: {EVAL_NEGATIVES}, seed: {EVAL_SEED}\n")


Initializing MBSRec Model

Model parameters: 371,603,400
Train users: 1313592  |  Items: 1855603
Val sessions (after filtering): 86727
Embed dim: 200, Heads: 4, Layers: 2
Maxlen: 70, Batch: 4112, Epochs: 50
Fixed eval users: 2000, neg/user: 99, seed: 2026



In [23]:
# Training Loop
T = 0.0
t0 = time.time()
best_metrics = {'Recall@10': 0.0, 'Recall@20': 0.0, 'NDCG@10': 0.0, 'NDCG@20': 0.0, 'HR@10': 0.0, 'HR@20': 0.0}

log_file = open(LOG_FILE, 'w')
log_file.write("="*80 + "\n")
log_file.write("MBSRec Training Log - Otto Validation Dataset\n")
log_file.write("Metrics: Recall@10, Recall@20, NDCG@10, NDCG@20, HR@10, HR@20\n")
log_file.write("="*80 + "\n\n")
log_file.flush()

print("\n" + "="*70)
print("Starting Training")
print("="*70)
print("\nMetrics: Recall@10, Recall@20, NDCG@10, NDCG@20, HR@10, HR@20\n")

# Early Stopping (patience=3, monitor=NDCG@20, min_delta=1e-4)
ES_PATIENCE = 3
ES_MIN_DELTA = 1e-4
es_counter = 0
es_best_ndcg = 0.0
early_stopped = False

if len(dataloader) == 0:
    raise RuntimeError('No training batches. Check strict filtering and dataset preparation.')

for epoch in range(1, config.num_epochs + 1):
    model.train()
    total_loss = 0
    epoch_start = time.time()
    
    pbar = tqdm(dataloader, desc=f"Epoch {epoch}/{config.num_epochs}")
    skipped_batches = 0
    for seq, pos, neg, beh_seq, beh_pos, pos_weight, neg_weight in pbar:
        seq = seq.to(DEVICE)
        pos = pos.to(DEVICE)
        neg = neg.to(DEVICE)
        beh_seq = beh_seq.to(DEVICE)
        beh_pos = beh_pos.to(DEVICE)
        pos_weight = pos_weight.to(DEVICE)
        neg_weight = neg_weight.to(DEVICE)

        mask = (seq != 0).float()

        optimizer.zero_grad(set_to_none=True)

        amp_context = autocast(device_type='cuda', enabled=True) if DEVICE == 'cuda' else nullcontext()
        with amp_context:
            enc_out = model(seq, beh_seq)

            seq_emb = enc_out
            pos_emb = model.target_item_embedding(pos, beh_pos)
            neg_emb = model.target_item_embedding(neg, beh_pos)

            batch_size, seq_len, emb_dim = seq_emb.shape
            seq_emb_flat = seq_emb.reshape(batch_size * seq_len, emb_dim)
            pos_emb_flat = pos_emb.reshape(batch_size * seq_len, emb_dim)
            neg_emb_flat = neg_emb.reshape(batch_size * seq_len, emb_dim)

            pos_logits = (seq_emb_flat * pos_emb_flat).sum(dim=-1)
            neg_logits = (seq_emb_flat * neg_emb_flat).sum(dim=-1)

            mask_flat = mask.reshape(-1)
            pos_weight_flat = pos_weight.reshape(-1)
            neg_weight_flat = neg_weight.reshape(-1)

            loss = criterion(pos_logits, neg_logits, pos_weight_flat, neg_weight_flat, mask_flat)

        if not torch.isfinite(loss):
            skipped_batches += 1
            continue

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item()
        pbar.set_postfix({'loss': f'{loss.item():.4f}', 'skip': skipped_batches})
    
    effective_steps = max(len(dataloader) - skipped_batches, 1)
    avg_loss = total_loss / effective_steps
    scheduler.step()

    epoch_time = time.time() - epoch_start
    T += epoch_time
    
    print(f"\n--- Epoch {epoch} Evaluation ---")
    print(f"Training Loss: {avg_loss:.4f}, Time: {epoch_time:.2f}s, Skipped batches: {skipped_batches}")
    
    # Evaluate on fixed users/candidates for reproducible epoch-to-epoch comparison.
    metrics, n_users = evaluate_full(
        model,
        val_seq,
        val_seq_types,
        val_gt,
        itemnum,
        config,
        DEVICE,
        ks=[10, 20],
        eval_users=fixed_eval_users,
        eval_negatives=fixed_eval_negatives,
    )
    
    log_line = f"Epoch {epoch}/{config.num_epochs} | "
    log_line += f"Loss: {avg_loss:.4f} | "
    log_line += f"R@10: {metrics['Recall@10']:.4f} | R@20: {metrics['Recall@20']:.4f} | "
    log_line += f"NDCG@10: {metrics['NDCG@10']:.4f} | NDCG@20: {metrics['NDCG@20']:.4f} | "
    log_line += f"HR@10: {metrics['HR@10']:.4f} | HR@20: {metrics['HR@20']:.4f} | "
    log_line += f"Time: {T:.2f}s"
    
    print(log_line)
    log_file.write(log_line + "\n")
    log_file.flush()
    
    # Save best model
    if metrics['NDCG@20'] >= best_metrics['NDCG@20']:
        best_metrics = metrics.copy()
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'metrics': best_metrics,
        }, os.path.join(OUTPUT_DIR, 'mbsrec_best.pth'))
        print(f"  ** Best model saved! NDCG@20: {best_metrics['NDCG@20']:.4f} **")

    # Early Stopping check
    if metrics['NDCG@20'] >= es_best_ndcg + ES_MIN_DELTA:
        es_best_ndcg = metrics['NDCG@20']
        es_counter = 0
    else:
        es_counter += 1
        print(f"  [EarlyStopping] No improvement for {es_counter}/{ES_PATIENCE} epochs  (best NDCG@20={es_best_ndcg:.4f})")
        if es_counter >= ES_PATIENCE:
            early_stopped = True
            log_file.write(f"Early stopping triggered at epoch {epoch}  (best NDCG@20={es_best_ndcg:.4f})\n")
            print(f"  >> Early stopping triggered at epoch {epoch}!")
            break

log_file.write("\n" + "="*80 + "\n")
if early_stopped:
    log_file.write("Training stopped early (patience exhausted).\n")
else:
    log_file.write("Training completed all epochs.\n")
log_file.write(f"Training Complete! Best Metrics:\n")
for k, v in best_metrics.items():
    log_file.write(f"  {k}: {v:.4f}\n")
log_file.write("="*80 + "\n")
log_file.close()

print("\n" + "="*70)
print("Training Complete!")
print("="*70)
print("\nBest Metrics:")
for k, v in best_metrics.items():
    print(f"  {k}: {v:.4f}")
print(f"\nLog saved to: {LOG_FILE}")


Starting Training

Metrics: Recall@10, Recall@20, NDCG@10, NDCG@20, HR@10, HR@20



Epoch 1/50: 100%|██████████| 320/320 [02:57<00:00,  1.80it/s, loss=0.1703, skip=0]



--- Epoch 1 Evaluation ---
Training Loss: 0.1867, Time: 177.92s, Skipped batches: 0
Epoch 1/50 | Loss: 0.1867 | R@10: 0.9210 | R@20: 0.9625 | NDCG@10: 0.7674 | NDCG@20: 0.7780 | HR@10: 0.9210 | HR@20: 0.9625 | Time: 177.92s
  ** Best model saved! NDCG@20: 0.7780 **


Epoch 2/50: 100%|██████████| 320/320 [02:58<00:00,  1.79it/s, loss=0.1305, skip=0]



--- Epoch 2 Evaluation ---
Training Loss: 0.1407, Time: 178.80s, Skipped batches: 0
Epoch 2/50 | Loss: 0.1407 | R@10: 0.9580 | R@20: 0.9790 | NDCG@10: 0.8618 | NDCG@20: 0.8672 | HR@10: 0.9580 | HR@20: 0.9790 | Time: 356.72s
  ** Best model saved! NDCG@20: 0.8672 **


...


--- Epoch 21 Evaluation ---
Training Loss: 0.0369, Time: 178.67s, Skipped batches: 0
Epoch 21/50 | Loss: 0.0369 | R@10: 0.9820 | R@20: 0.9880 | NDCG@10: 0.9482 | NDCG@20: 0.9497 | HR@10: 0.9820 | HR@20: 0.9880 | Time: 3753.49s
  [EarlyStopping] No improvement for 1/3 epochs  (best NDCG@20=0.9511)


Epoch 22/50: 100%|██████████| 320/320 [02:58<00:00,  1.79it/s, loss=0.0373, skip=0]



--- Epoch 22 Evaluation ---
Training Loss: 0.0359, Time: 178.32s, Skipped batches: 0
Epoch 22/50 | Loss: 0.0359 | R@10: 0.9825 | R@20: 0.9880 | NDCG@10: 0.9482 | NDCG@20: 0.9496 | HR@10: 0.9825 | HR@20: 0.9880 | Time: 3931.81s
  [EarlyStopping] No improvement for 2/3 epochs  (best NDCG@20=0.9511)


Epoch 23/50: 100%|██████████| 320/320 [02:58<00:00,  1.79it/s, loss=0.0357, skip=0]



--- Epoch 23 Evaluation ---
Training Loss: 0.0349, Time: 178.47s, Skipped batches: 0
Epoch 23/50 | Loss: 0.0349 | R@10: 0.9805 | R@20: 0.9870 | NDCG@10: 0.9489 | NDCG@20: 0.9506 | HR@10: 0.9805 | HR@20: 0.9870 | Time: 4110.29s
  [EarlyStopping] No improvement for 3/3 epochs  (best NDCG@20=0.9511)
  >> Early stopping triggered at epoch 23!

Training Complete!

Best Metrics:
  Recall@10: 0.9815
  Recall@20: 0.9875
  NDCG@10: 0.9496
  NDCG@20: 0.9511
  HR@10: 0.9815
  HR@20: 0.9875

Log saved to: /kaggle/working/mbsrec_output/training_log.txt


In [24]:
#Load test dataset
#Load test_df
test_df = _collect(pl.scan_parquet(str(TEST_PARQUET)).select(["session", "aid", "ts", "type"]))
test_df = test_df.sort(["session", "ts"]).group_by("session").agg([
                (pl.col("aid") + 1).sort_by(pl.col("ts")).alias("items"),  # shift +1
                pl.col("type").sort_by(pl.col("ts")).alias("types"),
            ]).select(["session", "items", "types"])
test_df.head(10)

session,items,types
i64,list[i64],list[str]
12899779,"[59626, 875855]","[""clicks"", ""clicks""]"
12899780,"[1142001, 582733, … 260306]","[""clicks"", ""clicks"", … ""clicks""]"
12899781,"[141737, 199009, … 918668]","[""clicks"", ""clicks"", … ""carts""]"
12899782,"[1669403, 1494781, … 1738522]","[""clicks"", ""clicks"", … ""clicks""]"
12899783,"[255298, 1114790, … 742845]","[""clicks"", ""clicks"", … ""clicks""]"
12899784,"[1036376, 1269953, … 588459]","[""clicks"", ""clicks"", … ""clicks""]"
12899785,"[1784452, 1169632, … 453906]","[""clicks"", ""clicks"", … ""clicks""]"
12899786,"[955253, 955253, … 955253]","[""clicks"", ""carts"", … ""clicks""]"
12899787,"[1682751, 1682751, … 740563]","[""clicks"", ""carts"", … ""clicks""]"


In [25]:
# Xử lý test dataset cho inference
class InferenceDataset(Dataset):
    def __init__(self, sessions, items_list, types_list, max_len):
        self.sessions = sessions
        self.items_list = items_list
        self.types_list = types_list
        self.max_len = max_len

    def __len__(self):
        return len(self.sessions)

    def __getitem__(self, idx):
        session = self.sessions[idx]
        items = self.items_list[idx]
        types = self.types_list[idx]
        
        seq = np.zeros(self.max_len, dtype=np.int32)
        beh_seq = np.zeros((self.max_len, 3), dtype=np.float32)
        
        # Chỉ lấy max_len items cuối cùng của chuỗi session
        items = items[-self.max_len:]
        types = types[-self.max_len:]
        
        idx_pos = self.max_len - 1
        for item, et_raw in zip(reversed(items), reversed(types)):
            seq[idx_pos] = item
            et = normalize_event_type(et_raw)
            beh_seq[idx_pos] = [
                1 if et == 'clicks' else 0,
                1 if et == 'carts'  else 0,
                1 if et == 'orders' else 0,
            ]
            idx_pos -= 1
            
        return session, torch.LongTensor(seq), torch.FloatTensor(beh_seq)

def run_mbsrec_inference(model, test_df, itemnum, config, top_k=20, batch_size=256, device='cuda'):
    model.eval()
    
    sessions = test_df["session"].to_list()
    items_list = [list(row) for row in test_df["items"].to_list()]
    types_list = [list(row) for row in test_df["types"].to_list()]
    
    dataset = InferenceDataset(sessions, items_list, types_list, config.maxlen)
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=4)
    
    # Pre-compute target embeddings cho tính toán batch
    # Tất cả items (1 -> itemnum) đều gán label target hành vi là 'orders' (0, 0, 1)
    with torch.no_grad():
        all_item_ids = torch.arange(1, itemnum + 1, dtype=torch.long, device=device)
        all_ctx = torch.tensor([[0.0, 0.0, 1.0]] * itemnum, dtype=torch.float, device=device)
        all_emb = model.target_item_embedding(all_item_ids, all_ctx)  # (itemnum, D)
        
    all_topk = []
    
    print(f"[Inference] {len(sessions):,} sessions | batch_size={batch_size}")
    
    with torch.no_grad():
        for session_ids, seq_batch, beh_batch in tqdm(loader, desc="Inference"):
            seq_batch = seq_batch.to(device)
            beh_batch = beh_batch.to(device)
            
            # Quét qua mô hình để lấy embedding chuỗi cuối cùng
            enc_out = model(seq_batch, beh_batch)
            seq_emb = enc_out[:, -1, :]  # (B, D)
            
            # Tính điểm số cho toàn bộ item = dot product seq_emb và all_emb
            scores = torch.matmul(seq_emb, all_emb.T)  # (B, itemnum)
            
            # Lấy top k
            _, top_indices = torch.topk(scores, k=top_k, dim=-1)  # (B, top_k)
            
            # Vì tập all_item_ids chạy từ 1 -> itemnum,
            # nên index 0 trong scores tương ứng với item_id=1.
            top_ids = all_item_ids[top_indices].cpu().numpy() - 1  # Trừ 1 để về ID gốc
            all_topk.append(top_ids)
            
    all_topk = np.concatenate(all_topk, axis=0)
    
    pred_df = pl.DataFrame({
        "session": sessions,
        "top_k_preds": all_topk.tolist(),
    })
    
    return pred_df

In [26]:
def run_mbsrec_inference_multi_behavior(model, test_df, itemnum, config, top_k=20, batch_size=256, device='cuda'):
    model.eval()
    
    sessions = test_df["session"].to_list()
    items_list = [list(row) for row in test_df["items"].to_list()]
    types_list = [list(row) for row in test_df["types"].to_list()]
    
    dataset = InferenceDataset(sessions, items_list, types_list, config.maxlen)
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=4)
    
    # Pre-compute target embeddings cho tính toán batch cho cả 3 hành vi
    with torch.no_grad():
        all_item_ids = torch.arange(1, itemnum + 1, dtype=torch.long, device=device)
        
        ctx_clicks = torch.tensor([[1.0, 0.0, 0.0]] * itemnum, dtype=torch.float, device=device)
        ctx_carts  = torch.tensor([[0.0, 1.0, 0.0]] * itemnum, dtype=torch.float, device=device)
        ctx_orders = torch.tensor([[0.0, 0.0, 1.0]] * itemnum, dtype=torch.float, device=device)
        
        emb_clicks = model.target_item_embedding(all_item_ids, ctx_clicks)  # (itemnum, D)
        emb_carts  = model.target_item_embedding(all_item_ids, ctx_carts)   # (itemnum, D)
        emb_orders = model.target_item_embedding(all_item_ids, ctx_orders)  # (itemnum, D)
        
    all_topk_clicks = []
    all_topk_carts = []
    all_topk_orders = []
    
    print(f"[Inference Multi-Behavior] {len(sessions):,} sessions | batch_size={batch_size}")
    
    with torch.no_grad():
        for session_ids, seq_batch, beh_batch in tqdm(loader, desc="Inference"):
            seq_batch = seq_batch.to(device)
            beh_batch = beh_batch.to(device)
            
            # Quét qua mô hình để lấy embedding chuỗi cuối cùng
            enc_out = model(seq_batch, beh_batch)
            seq_emb = enc_out[:, -1, :]  # (B, D)
            
            # Tính điểm số cho toàn bộ item = dot product seq_emb và all_emb của từng hành vi
            scores_clicks = torch.matmul(seq_emb, emb_clicks.T)  # (B, itemnum)
            scores_carts  = torch.matmul(seq_emb, emb_carts.T)   # (B, itemnum)
            scores_orders = torch.matmul(seq_emb, emb_orders.T)  # (B, itemnum)
            
            # Lấy top k
            _, top_idx_clicks = torch.topk(scores_clicks, k=top_k, dim=-1)
            _, top_idx_carts  = torch.topk(scores_carts, k=top_k, dim=-1)
            _, top_idx_orders = torch.topk(scores_orders, k=top_k, dim=-1)
            
            # Chuyển index về ID item thực
            all_topk_clicks.append(all_item_ids[top_idx_clicks].cpu().numpy() - 1)
            all_topk_carts.append(all_item_ids[top_idx_carts].cpu().numpy() - 1)
            all_topk_orders.append(all_item_ids[top_idx_orders].cpu().numpy() - 1)
            
    # Gộp list lại
    all_topk_clicks = np.concatenate(all_topk_clicks, axis=0)
    all_topk_carts  = np.concatenate(all_topk_carts, axis=0)
    all_topk_orders = np.concatenate(all_topk_orders, axis=0)
    
    pred_df = pl.DataFrame({
        "session": sessions,
        "clicks_preds": all_topk_clicks.tolist(),
        "carts_preds":  all_topk_carts.tolist(),
        "orders_preds": all_topk_orders.tolist(),
    })
    
    return pred_df

def create_submission_multi_behavior(pred_df, output_path="/kaggle/working/submission.csv"):
    """
    Tạo file submission dựa trên prediction DataFrame riêng rẽ cho từng behavior.
    """
    # Chuyển danh sách ID thành chuỗi cách nhau bởi khoảng trắng
    submission_df = pred_df.with_columns([
        pl.col("clicks_preds").list.eval(pl.element().cast(pl.String)).list.join(" ").alias("clicks_str"),
        pl.col("carts_preds").list.eval(pl.element().cast(pl.String)).list.join(" ").alias("carts_str"),
        pl.col("orders_preds").list.eval(pl.element().cast(pl.String)).list.join(" ").alias("orders_str"),
    ])
    
    rows = []
    for row in tqdm(submission_df.iter_rows(named=True),
                    total=len(submission_df),
                    desc="Building submission",
                    unit="session"):
        session = row["session"]
        
        rows.append({"session_type": f"{session}_clicks", "labels": row["clicks_str"]})
        rows.append({"session_type": f"{session}_carts",  "labels": row["carts_str"]})
        rows.append({"session_type": f"{session}_orders", "labels": row["orders_str"]})

    result_df = pl.DataFrame(rows)
    result_df.write_csv(output_path)

    print(f"[Submission Multi-Behavior] Saved → {output_path}")
    print(f"  Rows   : {len(result_df):,}")
    print(f"  Preview:")
    print(result_df.head(6))
    return result_df

In [27]:
# Khởi tạo model và load best state_dict
best_model_path = os.path.join(OUTPUT_DIR, 'mbsrec_best.pth')
# best_model_path ="/kaggle/input/models/trungnguyen2710t/mbsrec-model/pytorch/default/1/mbsrec_best.pth"
# itemnum=1855603
# Khởi tạo lại model với cùng tham số
infer_model = MBSRec(
    num_items=itemnum,
    num_behaviors=3,
    embed_dim=config.embed_dim,
    maxlen=config.maxlen,
    num_heads=config.num_heads,
    num_layers=config.num_layers,
    dropout=config.dropout
).to(DEVICE)

if os.path.exists(best_model_path):
    checkpoint = torch.load(best_model_path, map_location=DEVICE)
    infer_model.load_state_dict(checkpoint['model_state_dict'])
    infer_model.eval()
    print("Best model loaded successfully!")
else:
    print("Best model checkpoint not found, using trained model weights.")


Best model loaded successfully!


In [28]:
# Chạy inference
pred_df = run_mbsrec_inference(
    model=infer_model,
    test_df=test_df,
    itemnum=itemnum,
    config=config,
    top_k=20,
    batch_size=2056,  # Có thể tăng lên tuỳ RAM GPU.
    device=DEVICE
)

print(pred_df.head())

[Inference] 1,671,803 sessions | batch_size=2056


Inference: 100%|██████████| 814/814 [01:11<00:00, 11.35it/s]


shape: (5, 2)
┌──────────┬──────────────────────────────┐
│ session  ┆ top_k_preds                  │
│ ---      ┆ ---                          │
│ i64      ┆ list[i64]                    │
╞══════════╪══════════════════════════════╡
│ 12899779 ┆ [1435662, 175833, … 528395]  │
│ 12899780 ┆ [409620, 231487, … 884502]   │
│ 12899781 ┆ [999262, 43313, … 362490]    │
│ 12899782 ┆ [1836493, 585521, … 1421190] │
│ 12899783 ┆ [715263, 959208, … 1030727]  │
└──────────┴──────────────────────────────┘


In [29]:
def create_submission(pred_df, output_path="/kaggle/working/submission.csv"):
    """
    Tạo file submission theo định dạng Kaggle OTTO:
        session_type          | labels
        12899779_clicks       | 129004 126836 118524 ...
        12899779_carts        | 129004 126836 118524 ...
        12899779_orders       | 129004 126836 118524 ...
    """
    submission_df = (
        pred_df
        .with_columns(
            pl.col("top_k_preds")
               .list.eval(pl.element().cast(pl.String))
               .list.join(" ")
               .alias("labels")
        )
        .select(["session", "labels"])
    )

    rows = []
    for row in tqdm(submission_df.iter_rows(named=True),
                    total=len(submission_df),
                    desc="Building submission",
                    unit="session"):
        session    = row["session"]
        labels_str = row["labels"]
        for action_type in ["clicks", "carts", "orders"]:
            rows.append({
                "session_type": f"{session}_{action_type}",
                "labels"      : labels_str,
            })

    result_df = pl.DataFrame(rows)
    result_df.write_csv(output_path)

    print(f"[Submission] Saved → {output_path}")
    print(f"  Rows   : {len(result_df):,}")
    print(f"  Preview:")
    print(result_df.head(6))
    return result_df


In [30]:
# Bước 4: Tạo submission
submission = create_submission(
    pred_df     = pred_df,
    output_path = "/kaggle/working/mbsrec_submission.csv",
)

Building submission: 100%|██████████| 1671803/1671803 [00:01<00:00, 1223484.03session/s]


[Submission] Saved → /kaggle/working/mbsrec_submission.csv
  Rows   : 5,015,409
  Preview:
shape: (6, 2)
┌─────────────────┬─────────────────────────────────┐
│ session_type    ┆ labels                          │
│ ---             ┆ ---                             │
│ str             ┆ str                             │
╞═════════════════╪═════════════════════════════════╡
│ 12899779_clicks ┆ 1435662 175833 1497089 240800 … │
│ 12899779_carts  ┆ 1435662 175833 1497089 240800 … │
│ 12899779_orders ┆ 1435662 175833 1497089 240800 … │
│ 12899780_clicks ┆ 409620 231487 618310 1502122 8… │
│ 12899780_carts  ┆ 409620 231487 618310 1502122 8… │
│ 12899780_orders ┆ 409620 231487 618310 1502122 8… │
└─────────────────┴─────────────────────────────────┘


In [31]:
# Chạy inference và tạo submission đa hành vi (Multi-Behavior)
pred_df_multi = run_mbsrec_inference_multi_behavior(
    model=infer_model,
    test_df=test_df,
    itemnum=itemnum,
    config=config,
    top_k=20,
    batch_size=2056,  # Tùy chỉnh batch_size theo VRAM GPU của bạn
    device=DEVICE
)

# In thử 5 dòng đầu
print("Prediction DataFrame (Multi-Behavior):")
print(pred_df_multi.head())

# Tạo file submission định dạng Kaggle
final_sub_multi = create_submission_multi_behavior(
    pred_df_multi, 
    output_path="/kaggle/working/submission_multi.csv"
)

[Inference Multi-Behavior] 1,671,803 sessions | batch_size=2056


Inference: 100%|██████████| 814/814 [03:04<00:00,  4.41it/s]


Prediction DataFrame (Multi-Behavior):
shape: (5, 4)
┌──────────┬─────────────────────────────┬────────────────────────────┬────────────────────────────┐
│ session  ┆ clicks_preds                ┆ carts_preds                ┆ orders_preds               │
│ ---      ┆ ---                         ┆ ---                        ┆ ---                        │
│ i64      ┆ list[i64]                   ┆ list[i64]                  ┆ list[i64]                  │
╞══════════╪═════════════════════════════╪════════════════════════════╪════════════════════════════╡
│ 12899779 ┆ [1435662, 175833, … 528395] ┆ [1435662, 175833, …        ┆ [1435662, 175833, …        │
│          ┆                             ┆ 528395]                    ┆ 528395]                    │
│ 12899780 ┆ [409620, 231487, … 884502]  ┆ [409620, 231487, … 884502] ┆ [409620, 231487, … 884502] │
│ 12899781 ┆ [999262, 43313, … 362490]   ┆ [999262, 43313, … 362490]  ┆ [999262, 43313, … 362490]  │
│ 12899782 ┆ [1836493, 585521, …      

Building submission: 100%|██████████| 1671803/1671803 [00:06<00:00, 244601.15session/s]


[Submission Multi-Behavior] Saved → /kaggle/working/submission_multi.csv
  Rows   : 5,015,409
  Preview:
shape: (6, 2)
┌─────────────────┬─────────────────────────────────┐
│ session_type    ┆ labels                          │
│ ---             ┆ ---                             │
│ str             ┆ str                             │
╞═════════════════╪═════════════════════════════════╡
│ 12899779_clicks ┆ 1435662 175833 1497089 240800 … │
│ 12899779_carts  ┆ 1435662 175833 1497089 240800 … │
│ 12899779_orders ┆ 1435662 175833 1497089 240800 … │
│ 12899780_clicks ┆ 409620 231487 618310 1502122 8… │
│ 12899780_carts  ┆ 409620 231487 618310 1502122 8… │
│ 12899780_orders ┆ 409620 231487 618310 1502122 8… │
└─────────────────┴─────────────────────────────────┘


In [32]:
type_to_onehot = [
    [1.0, 0.0, 0.0], # clicks
    [0.0, 1.0, 0.0], # carts
    [0.0, 0.0, 1.0], # orders
]
for u in list(val_seq.keys()):
    items = val_seq[u]
    types = val_seq_types[u]
    try:
        for i in range(len(items) - 1, -1, -1):
            _ = type_to_onehot[types[i]]
    except Exception as e:
        print(f"Error on user {u}: {e}")
        print(f"len(items)={len(items)}, len(types)={len(types)}")
        if len(items) == len(types):
            print(f"types: {types}")
        elif i >= len(types):
            print(f"i={i}, len(types)={len(types)}")
        break